# Notebook 07 — Kizomba Tutor (Phase 26 + Phase 29b)

Per-track kizomba coaching: load → analyze with `dance_style="kizomba"` →
show the beat overlay and per-section beat-clarity → ask Gemma
(small local model, recommended `gemma4:e4b` via Ollama) for short
learner-facing notes per phase using the `kizomba_tutor` prompt. Phase 29b
adds an optional polished rewrite pass at the end.

Honest scope reminders:
* Kizomba downbeat / `"1"` detection is **out of scope** — the prompt forbids naming a `"1"` position.
* Where the analysis reports `beat: subtle`, the tutor warns the learner instead of pretending the pulse is locked.
* The 5 tracks here are the kizomba tap-reference set. Charbel is the hardest case (F=0.399 on taps) — expect more `beat: subtle` phases and honest recovery coaching.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

from IPython.display import HTML, display

from rytmi import analyze, load_audio
from rytmi import viz as rytmi_viz
from rytmi.llm import explain_rhythm, load_cloud_model, polish_kizomba_tutor_output
from rytmi.prompts import QUESTION_KIZOMBA_TUTOR

# Local Gemma E4B via Ollama is the project default. Override with
# RYTMI_API_BASE_URL / RYTMI_API_KEY env vars (e.g. for a cloud endpoint).
CLOUD_BASE_URL = os.environ.get("RYTMI_API_BASE_URL", "http://localhost:11434/v1")
CLOUD_API_KEY = os.environ.get("RYTMI_API_KEY", "ollama")
CLOUD_MODEL_ID = os.environ.get("RYTMI_MODEL_ID", "google/gemma-4-26b-a4b-it")#"gemma4:e4b")

EVAL = Path("../data/songs/eval_set/kizomba")
# Phase 29b — the 5 tap-reference tracks used for ground-truth eval.
# Charbel is the known hard case (F=0.399, wrong-phase recovery issues);
# capturing the *honest* tutor behavior on it — including how it handles
# `beat: subtle` sections — is the kind of "surface uncertainty instead of
# pretending confidence" example the demo benefits from.
TRACKS: list[Path] = [
    EVAL / "Baila_Kizomba_Amor [XG11YxMWgaI].mp3",
    EVAL / "Filomena_Maricoa_-_Teu_Toque_Official_Video [sKbWzD0mt6I].mp3",
    EVAL / "Charbel_-_E_Magia_Official_Video_4K [QkfyDj8aJRM].mp3",
    EVAL / "Criola [kCOie6jQXag].mp3",
    EVAL / "MIKA_MENDES_-_MAGICO_2011 [ZM1GnUImrPw].mp3",
]

PIXELS_PER_SECOND = 80

## Load the Gemma client (cheap — just an HTTP wrapper)

In [ ]:
processor, model = load_cloud_model(
    base_url=CLOUD_BASE_URL,
    api_key=CLOUD_API_KEY,
    model_id=CLOUD_MODEL_ID,
)
#print(f"endpoint: {CLOUD_BASE_URL}  model: {CLOUD_MODEL_ID}")

## Per-track helper: analyze, draw, ask Gemma

In [ ]:
def tutor(path: Path) -> tuple[str, str]:
    """Run analyze + viz + one-pass tutor for ``path``. Returns (name, draft)."""
    name = path.stem.split(" [")[0]
    print(f"\n========== {name} ==========")
    audio = load_audio(path)
    a = analyze(audio, dance_style="kizomba")
    print(f"  tempo={a.tempo:.1f} BPM | beats={len(a.beats.times)} | sections={len(a.sections)}")
    for sec in a.sections:
        bc = sec.rhythm_features.beat_clarity if sec.rhythm_features else float('nan')
        print(f"    {sec.label:8s} {sec.start_s:6.1f}s-{sec.end_s:6.1f}s  beat_clarity={bc:.2f}")
    display(HTML(rytmi_viz.interactive_timeline(
        a, title=f"{name} — kizomba beats",
        pixels_per_second=PIXELS_PER_SECOND,
    )))
    print("\n--- Gemma kizomba_tutor (one-pass) ---")
    draft = explain_rhythm(a, QUESTION_KIZOMBA_TUTOR, processor, model)
    print(draft)
    return name, draft

## Run on the 5 kizomba tap-reference tracks

Charbel is the hardest case — F=0.399 on tap-reference; the tutor should be honest about `beat: subtle` sections rather than overclaiming danceability.

In [ ]:
DRAFTS: dict[str, str] = {}
for p in TRACKS:
    if p.exists():
        name, draft = tutor(p)
        DRAFTS[name] = draft
    else:
        print(f"missing: {p}")

## Phase 29b — optional polished rewrite

Run a second LLM pass over each one-pass draft using `polish_kizomba_tutor_output`. The rewrite preserves every P# header, time span, and beat tag from the draft but tightens the coaching text against a stricter rubric. Compare side-by-side with the one-pass baseline above.

In [ ]:
# Phase 29b — optional second-pass polish over each one-pass draft.
#
# `polish_kizomba_tutor_output` runs an extra LLM call against a stricter
# coaching rubric (see KIZOMBA_TUTOR_POLISH_SYSTEM and
# build_kizomba_tutor_polish_prompt in rytmi.prompts). It preserves every
# P# header, time span, and beat tag from the draft but rewrites the
# coaching text. Useful for deciding whether the polish pass is worth
# adding to the app/demo flow — keep one-pass output as the documented
# baseline; polish is opt-in.

for name, draft in DRAFTS.items():
    print(f"\n========== {name} ==========")
    print("--- one-pass draft ---")
    print(draft.strip())
    print("\n--- polished rewrite ---")
    print(polish_kizomba_tutor_output(draft, processor, model, track_name=name))